# param-grad-access — ex3: classify each grad as nan, zero, or ok — three-bucket health diagnostic

> Procedural drill from [Delta Drills](https://delta-drills.vercel.app).
> Atom: `param-grad-access`. Running the final beacon cell reports progress against the `PyTorch: param.grad access` subtopic.

**Why this is a Colab exercise.** This standalone exercises material the Delta Drills flashcards can't deliver on their own — interactive tensor execution, visualization, or multi-step debugging. Read the prompt, fill in the function body, run the test cell, then run the beacon at the bottom.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)
import torch.nn as nn

## Connect to Delta Drills

Paste your Delta Drills auth token below so this drill can report progress on the `PyTorch: param.grad access` subtopic. Copy it from your Delta Drills account page.

This standalone exercises the atom **`param-grad-access`** (exercise 3). Completion fires the beacon at the bottom.

In [ ]:
# === Delta Drills auth ===
DD_TOKEN = ""  # paste your token here, then run this cell
DD_ATOM_ID = "param-grad-access"
DD_SUBTOPIC = "PyTorch: param.grad access"
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## Grad health classification — three buckets per param

Ex1 applied the SGD step (`p.data -= lr * p.grad`). Ex2 collected per-
param L2 norms. The deepening move is a DIAGNOSTIC view: for each
parameter, ask `is this grad healthy?` and bucket it into one of three
categories:

| Bucket   | Condition                                       | Meaning                                |
|----------|--------------------------------------------------|----------------------------------------|
| `'nan'`  | `t.isnan(p.grad).any()` or `t.isinf(p.grad).any()` | numerical blow-up — abort the step    |
| `'zero'` | all-finite AND `p.grad.abs().max() == 0`         | dead neuron / disconnected layer       |
| `'ok'`   | all-finite AND any nonzero element               | healthy — training can proceed         |

Params with `.grad is None` are SKIPPED — they didn't participate in
the forward pass, so there's nothing to diagnose.

**Why classify rather than norm.** A grad of L2-norm `1e-12` and a grad
of `nan` are both 'small' under the norm view but have completely
different operational meanings. The three-bucket classification gives
you an unambiguous trigger surface — `'nan'` → skip step + maybe halve
lr; `'zero'` → check upstream layer connectivity; `'ok'` → proceed.

**Order of checks matters.** Test `nan/inf` FIRST. A tensor of all-NaN
satisfies `abs().max() == nan != 0`, so the zero-check is meaningless
until NaN is ruled out.

### Exercise 3 — classify each grad as nan, zero, or ok — three-bucket health diagnostic

> ```yaml
> Difficulty: 🔴🔴🔴⚪⚪
> Bloom level: Analyze
> LO: Analyze each parameter's `.grad` tensor and classify it as `'nan'` (any NaN/Inf), `'zero'` (all finite + all zero), or `'ok'` (all finite + at least one nonzero), skipping params whose `.grad is None`.
> Keywords: nan, inf, zero-grad, diagnostic, health
> ```

**KCs targeted:** `param-grad-none-guard-skip`, `nan-before-zero-check-order`

Implement `ex3_classify_grad_health(model)`. Return a `dict[str, str]` mapping each parameter NAME (from `model.named_parameters()`) to one of three labels:

- `'nan'` — if `t.isnan(p.grad).any()` OR `t.isinf(p.grad).any()`. This check MUST come first (a tensor of all-NaN reads as nonzero to the zero-check, so order matters).
- `'zero'` — all finite AND `p.grad.abs().max().item() == 0.0`.
- `'ok'` — all finite AND any element nonzero.

Parameters whose `.grad is None` MUST be OMITTED from the dict (not present as `'none'`, just absent). This matches the ex2 convention.

Required ordering:
1. None-guard: skip if `p.grad is None`.
2. NaN/Inf check FIRST.
3. Zero check second.
4. Default to `'ok'`.

Return type: `dict[str, str]`. The test feeds a model with a mix of all four cases (none, nan, zero, ok) and asserts exact labels per parameter.

In [ ]:
def ex3_classify_grad_health(model) -> dict:
    """Return {param-name: nan/zero/ok}, skipping params with .grad is None."""
    raise NotImplementedError()


def _test_ex3():
    def _test_ex3():
        import torch.nn as nn

        # Build a model with four named params so we can hand-set each .grad.
        class FourParams(nn.Module):
            def __init__(self):
                super().__init__()
                self.p_none = nn.Parameter(t.zeros(3))
                self.p_nan  = nn.Parameter(t.zeros(3))
                self.p_zero = nn.Parameter(t.zeros(3))
                self.p_ok   = nn.Parameter(t.zeros(3))

        model = FourParams()
        # leave model.p_none.grad as None
        model.p_nan.grad  = t.tensor([1.0, float('nan'), 3.0])
        model.p_zero.grad = t.zeros(3)
        model.p_ok.grad   = t.tensor([0.1, 0.2, 0.3])

        out = ex3_classify_grad_health(model)
        assert isinstance(out, dict), f'must return dict, got {type(out).__name__}'
        assert 'p_none' not in out, f'p_none has grad=None; must be OMITTED, got {out}'
        assert out.get('p_nan') == 'nan', f'p_nan should be nan; got {out}'
        assert out.get('p_zero') == 'zero', f'p_zero should be zero; got {out}'
        assert out.get('p_ok') == 'ok', f'p_ok should be ok; got {out}'
        assert len(out) == 3, f'should have 3 entries (p_none omitted); got {out}'

        # === Inf is classified as 'nan' (same bucket — numerical blow-up) ===
        model.p_ok.grad = t.tensor([1.0, float('inf'), 2.0])
        out = ex3_classify_grad_health(model)
        assert out['p_ok'] == 'nan', f'inf grad should bucket to nan; got {out}'

        # === Negative inf also bucketed as 'nan' ===
        model.p_ok.grad = t.tensor([1.0, float('-inf'), 2.0])
        out = ex3_classify_grad_health(model)
        assert out['p_ok'] == 'nan', f'-inf grad should bucket to nan; got {out}'

        # === All-NaN tensor: zero-check would erroneously hit, so order is the test ===
        model.p_ok.grad = t.tensor([float('nan'), float('nan'), float('nan')])
        out = ex3_classify_grad_health(model)
        assert out['p_ok'] == 'nan', f'all-NaN must classify as nan (order test); got {out}'

        # === A grad with a single nonzero element is 'ok', not 'zero' ===
        model.p_ok.grad = t.tensor([0.0, 0.0, 1e-8])
        out = ex3_classify_grad_health(model)
        assert out['p_ok'] == 'ok', f'tiny nonzero element is still ok; got {out}'

        # === Empty model (no params) returns empty dict ===
        class Empty(nn.Module):
            pass
        assert ex3_classify_grad_health(Empty()) == {}, 'empty model should yield empty dict'

        # === Nested model: dotted names from named_parameters preserved ===
        class Nested(nn.Module):
            def __init__(self):
                super().__init__()
                self.fc = nn.Linear(2, 2)
        nested = Nested()
        nested.fc.weight.grad = t.tensor([[1.0, 2.0], [3.0, 4.0]])
        nested.fc.bias.grad = t.zeros(2)
        out = ex3_classify_grad_health(nested)
        assert out == {'fc.weight': 'ok', 'fc.bias': 'zero'}, out
        print('ex3 ok')

    _test_ex3()
    _dd_passed.add('ex3')
    print("ex3 ✓")

_test_ex3()

<details><summary>Solution</summary>

```python
def ex3_classify_grad_health(model):
    out = {}
    for name, p in model.named_parameters():
        if p.grad is None:
            continue
        # ORDER MATTERS: nan/inf first — an all-NaN tensor would otherwise
        # fail the zero-check ambiguously (nan != 0 but also nan != nonzero).
        if t.isnan(p.grad).any().item() or t.isinf(p.grad).any().item():
            out[name] = 'nan'
        elif p.grad.abs().max().item() == 0.0:
            out[name] = 'zero'
        else:
            out[name] = 'ok'
    return out
```

**`.any().item()` is the safe coerce.** `t.isnan(x).any()` returns a 0-D tensor that's truthy in Python — but mixing tensor-bools with `or`/`and` triggers DeprecationWarnings on some torch versions. Coercing to Python bool with `.item()` is explicit.

**Why combine nan + inf into one bucket.** Both indicate numerical pathology in the same way for an optimizer — the step should be skipped, the learning rate possibly cut, the run possibly aborted. Two separate buckets would just add a needless branch in caller code.

**`abs().max() == 0` over `(p.grad == 0).all()`.** Both work for finite tensors, but `abs().max()` is one reduction kernel vs an equality-broadcast + reduce. Marginal but consistent with the PyTorch internals style.
</details>

## Report completion

Run the cell below to send your progress to Delta Drills. The beacon fires only if the test cell above passed.

In [ ]:
# === Delta Drills completion beacon ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'ex3'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'procedural-drill:{DD_ATOM_ID}:ex3',
        'subtopics': [DD_SUBTOPIC],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported {DD_ATOM_ID} (subtopic={DD_SUBTOPIC!r})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()